In [1]:
import sys
sys.path.insert(0, "/home/ibm-user/shared/safecircle")   # so "from src.explain import ..." works
sys.path.insert(0, "/home/ibm-user")                      # so Lab 2's Granite helper can be imported later

from src.explain import redact, explain

print(redact("Grandma it's me, call me at +1-555-0142"))
print(explain("test", {"risk": 0.9, "action": "hold_and_alert", "reasons": ["asks for gift cards"]}))

Grandma it's me, call me at [PHONE]
stub explanation - real version coming


In [7]:
import re
import pandas as pd

# Family words are the scam clue, not private data, so we never hide them
FAMILY_WORDS = {"Grandma", "Grandpa", "Grandmother", "Grandfather", "Nana", "Papa",
                "Mom", "Dad", "Mother", "Father", "Sir", "Madam", "Customer"}

# Names we know show up in our data. Add any others you spot in the test output.
KNOWN_NAMES = ["Rosa", "Walter", "Linda"]

CARD_RE  = re.compile(r"\b\d(?:[ -]?\d){11,18}\b")                    # 12-19 digits
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")             # name@site.com
LINK_RE  = re.compile(r"(?:https?://|www\.)\S+"
                      r"|\b[\w-]+(?:\.[\w-]+)*\.(?:com|net|org|xyz|info|io|co|us|ly)\b(?:/\S*)?")
PHONE_RE = re.compile(r"(?:\+?1[-.\s]?)?(?:\(?\d{3}\)?[-.\s]?)?\b\d{3}[-.\s]\d{4}\b")
GREETING_RE = re.compile(r"\b(Hi|Hello|Hey|Dear|this is|This is|it's|It's|it is|It is)\s+([A-Z][a-z]+)")
TITLE_RE   = re.compile(r"\b((?:Dr|Mr|Mrs|Ms|Officer|Agent|Deputy|Sgt)\.?)\s+[A-Z][a-z]+")
ADDRESS_RE = re.compile(r"\b\d{1,5}\s+(?:[A-Z][a-z]+\s+){1,3}(?:St|Street|Ave|Avenue|Rd|Road|Blvd|Dr|Drive|Ln|Lane|Way|Ct|Court)\b")

def redact(text: str) -> str:
    """Hide card numbers, emails, links, phone numbers and names before text reaches the language model."""
    text = CARD_RE.sub("[CARD]", text)      # 1st: so the phone rule can't grab part of a card number
    text = EMAIL_RE.sub("[EMAIL]", text)    # 2nd: before links, so an email isn't split up
    text = LINK_RE.sub("[LINK]", text)
    text = PHONE_RE.sub("[PHONE]", text)
    text = ADDRESS_RE.sub("[ADDRESS]", text)
    text = TITLE_RE.sub(r"\1 [NAME]", text)          # "Dr. Patel" -> "Dr. [NAME]"

    def hide_name(match):
        if match.group(2) in FAMILY_WORDS:
            return match.group(0)                 # keep "Hi Grandma"
        return f"{match.group(1)} [NAME]"         # "It's Tyler" -> "It's [NAME]"
    text = GREETING_RE.sub(hide_name, text)

    for name in KNOWN_NAMES:
        text = re.sub(rf"\b{name}\b", "[NAME]", text)
    return text


print(redact("Appointment with Dr. Patel at 123 Main St. Call 555-0199. Hi Linda!"))

Appointment with Dr. [NAME] at [ADDRESS]. Call [PHONE]. Hi [NAME]!


In [8]:
extra = pd.read_csv("/home/ibm-user/shared/safecircle/data/extra_messages.csv")
scen  = pd.read_csv("/home/ibm-user/shared/safecircle/scenarios/messages.csv")
texts = pd.concat([extra["text"], scen["text"]], ignore_index=True)

results = pd.DataFrame({"before": texts, "after": texts.apply(redact)})
changed = results[results["before"] != results["after"]]
print(f"{len(results)} texts tested, {len(changed)} changed\n")

for _, row in changed.iterrows():
    print("BEFORE:", row["before"])
    print("AFTER: ", row["after"], "\n")

print("=== UNCHANGED (skim these for any missed name or number) ===")
for t in results.loc[results["before"] == results["after"], "before"]:
    print("-", t)

45 texts tested, 15 changed

BEFORE: This is Officer Daniels. Your grandson was arrested and needs $3,500 for bail. Pay with Apple gift cards and read me the numbers. This case is confidential.
AFTER:  This is [NAME] [NAME]. Your grandson was arrested and needs $3,500 for bail. Pay with Apple gift cards and read me the numbers. This case is confidential. 

BEFORE: URGENT: Your bank account has been locked due to suspicious activity. Verify your identity now at secure-acct-verify.xyz or your account will be closed.
AFTER:  URGENT: Your bank account has been locked due to suspicious activity. Verify your identity now at [LINK] or your account will be closed. 

BEFORE: Wells Fargo Alert: unusual sign-in detected. Confirm your card number and PIN at wf-secure-login.xyz within 24 hours to avoid suspension.
AFTER:  Wells Fargo Alert: unusual sign-in detected. Confirm your card number and PIN at [LINK] within 24 hours to avoid suspension. 

BEFORE: Your Zelle payment of $950 failed. To get a 

In [9]:
suspicious = results[results["after"].str.contains(r"\d{3}[-.\s]\d{4}|\d{7,}|@|https?://|\.xyz", regex=True)]
print(len(suspicious), "texts still look private")
suspicious["after"].tolist()

0 texts still look private


[]

In [2]:
from src.explain import redact
print(redact("Appointment with Dr. Patel at 123 Main St. Call 555-0199. Hi Linda!"))

Appointment with Dr. [NAME] at [ADDRESS]. Call [PHONE]. Hi [NAME]!
